In [1]:
import os

os.chdir(os.path.join(os.path.abspath("."),".."))
os.getcwd()

'/home/harim/Desktop/kaist/DS801/StockMixer'

In [15]:
import numpy as np
import pickle

In [16]:
data_path = './dataset'
market_name = 'NASDAQ'

dataset_path = os.path.join(data_path, market_name)
with open(os.path.join(dataset_path, "eod_data.pkl"), "rb") as f:
    eod_data = pickle.load(f)
with open(os.path.join(dataset_path, "mask_data.pkl"), "rb") as f:
    mask_data = pickle.load(f)
with open(os.path.join(dataset_path, "gt_data.pkl"), "rb") as f:
    gt_data = pickle.load(f)
with open(os.path.join(dataset_path, "price_data.pkl"), "rb") as f:
    price_data = pickle.load(f)

In [18]:
eod_data.shape, mask_data.shape, gt_data.shape, price_data.shape

((1026, 1245, 5), (1026, 1245), (1026, 1245), (1026, 1245))

In [19]:
stock_num = eod_data.shape[0]
fea_num = eod_data.shape[2]
lookback_length = 16

data_counts = eod_data.shape[1] - lookback_length + 1
eod_data_sliced = np.zeros((data_counts-1, stock_num, lookback_length, fea_num))
mask_data_sliced = np.zeros((data_counts-1, stock_num))
gt_data_sliced = np.zeros((data_counts-1, stock_num))
price_data_sliced = np.zeros((data_counts-1, stock_num))

for i in range(data_counts-1):
    eod_data_sliced[i] = eod_data[:, i:i+lookback_length, :]
    mask_data_sliced[i] = mask_data[:, i:i+lookback_length+1].min(axis=1)
    gt_data_sliced[i] = gt_data[:, i+lookback_length]
    price_data_sliced[i] = price_data[:, i+lookback_length]

In [20]:
eod_data_sliced.shape, mask_data_sliced.shape, gt_data_sliced.shape, price_data_sliced.shape

((1229, 1026, 16, 5), (1229, 1026), (1229, 1026), (1229, 1026))

In [21]:
test_days = 273
val_days = 252
train_days = data_counts - test_days - val_days

train_eod_data = eod_data_sliced[:train_days]
train_mask_data = mask_data_sliced[:train_days]
train_gt_data = gt_data_sliced[:train_days]
train_price_data = price_data_sliced[:train_days]

val_eod_data = eod_data_sliced[train_days:train_days+val_days]
val_mask_data = mask_data_sliced[train_days:train_days+val_days]
val_gt_data = gt_data_sliced[train_days:train_days+val_days]
val_price_data = price_data_sliced[train_days:train_days+val_days]

test_eod_data = eod_data_sliced[train_days+val_days:]
test_mask_data = mask_data_sliced[train_days+val_days:]
test_gt_data = gt_data_sliced[train_days+val_days:]
test_price_data = price_data_sliced[train_days+val_days:]

In [22]:
train_eod_data.shape, train_mask_data.shape, train_gt_data.shape, train_price_data.shape

((705, 1026, 16, 5), (705, 1026), (705, 1026), (705, 1026))

In [23]:
test_eod_data.shape, test_mask_data.shape, test_gt_data.shape, test_price_data.shape

((272, 1026, 16, 5), (272, 1026), (272, 1026), (272, 1026))

In [24]:
import torch
train_eod_data = torch.tensor(train_eod_data, dtype=torch.float32)
train_mask_data = torch.tensor(train_mask_data, dtype=torch.float32)
train_gt_data = torch.tensor(train_gt_data, dtype=torch.float32)
train_price_data = torch.tensor(train_price_data, dtype=torch.float32)

In [25]:
from torch.utils.data import DataLoader, TensorDataset

train_dataset = TensorDataset(train_eod_data, train_mask_data, train_gt_data, train_price_data)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

In [26]:
a,b,c,d = next(iter(train_loader))

In [28]:
a = a.squeeze()
b = b.squeeze().unsqueeze(-1)
c = c.squeeze().unsqueeze(-1)
d = d.squeeze().unsqueeze(-1)
a.shape, b.shape, c.shape, d.shape

(torch.Size([1026, 16, 5]),
 torch.Size([1026, 1]),
 torch.Size([1026, 1]),
 torch.Size([1026, 1]))

In [29]:
epochs = 100
valid_index = 756
test_index = 1008
market_num = 20
steps = 1
learning_rate = 0.001
alpha = 0.1
scale_factor = 3
activation = 'GELU'

In [6]:
from src.model import StockMixer, get_loss
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = StockMixer(
    stocks=stock_num,
    time_steps=lookback_length,
    channels=fea_num,
    market=market_num,
    scale=scale_factor
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

best_valid_loss = np.inf
best_valid_perf = None
best_test_perf = None
batch_offsets = np.arange(start=0, stop=valid_index, dtype=int)



In [7]:
np.random.shuffle(batch_offsets)
tra_loss = 0.0
tra_reg_loss = 0.0
tra_rank_loss = 0.0

In [ ]:
import random

def get_batch(offset=None):
    if offset is None:
        offset = random.randrange(0, valid_index)
    seq_len = lookback_length
    mask_batch = mask_data[:, offset: offset + seq_len + steps]
    mask_batch = np.min(mask_batch, axis=1)
    return (
        eod_data[:, offset:offset + seq_len, :],
        np.expand_dims(mask_batch, axis=1),
        np.expand_dims(price_data[:, offset + seq_len - 1], axis=1),
        np.expand_dims(gt_data[:, offset + seq_len + steps - 1], axis=1))

